# Tree Prediction Distributions across Temperature Bins

For the latest EBV model (`rf_gaia_teff_corrected_log_ebv_optuna_20260401_201113`),
sample one random test object per 1000 K bin and visualize:
- Histogram of individual-tree predictions
- Fitted 5-component GMM overlay (total + individual components)
- True temperature value

In [ ]:
import sys
sys.path.insert(0, '..')

import joblib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from src.visualization.tree_distributions import (
    plot_tree_distribution_single,
    plot_tree_distributions_grid,
)
from src.visualization.validation_plots import save_figure

%matplotlib inline

## Load model and test predictions

In [ ]:
MODEL_ID = "rf_gaia_teff_corrected_log_ebv_optuna_20260401_201113"
MODELS_DIR = Path("../models")

model = joblib.load(MODELS_DIR / f"{MODEL_ID}.pkl")
test_pred = pd.read_parquet(MODELS_DIR / f"{MODEL_ID}_test_predictions.parquet")
with open(MODELS_DIR / f"{MODEL_ID}_metadata.json") as f:
    metadata = json.load(f)

features = metadata["features"]
target_transform = metadata.get("target_transform", "none")

print(f"Model: {MODEL_ID}")
print(f"Trees: {len(model.estimators_)}")
print(f"Features: {features}")
print(f"Target transform: {target_transform}")
print(f"Test samples: {len(test_pred):,}")
print(f"Teff range: {test_pred['y_true'].min():.0f} – {test_pred['y_true'].max():.0f} K")

## Sample one random object per 1000 K temperature bin

Bins span from 3000 K to the maximum true Teff in 1000 K steps.
We pick one random object from each bin that has at least one member.

In [ ]:
BIN_STEP = 1000  # K
teff_min = 3000
teff_max = int(np.ceil(test_pred["y_true"].max() / BIN_STEP) * BIN_STEP)
bin_edges = np.arange(teff_min, teff_max + BIN_STEP, BIN_STEP)

rng = np.random.RandomState(42)
sampled_rows = []
bin_labels = []

for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    mask = (test_pred["y_true"] >= lo) & (test_pred["y_true"] < hi)
    candidates = test_pred[mask]
    if len(candidates) == 0:
        continue
    row = candidates.sample(1, random_state=rng)
    sampled_rows.append(row)
    bin_labels.append(f"{lo/1000:.0f}–{hi/1000:.0f} kK  (n={len(candidates):,})")

sampled = pd.concat(sampled_rows).reset_index(drop=True)
print(f"Selected {len(sampled)} objects from {len(bin_edges)-1} bins")
for i, lbl in enumerate(bin_labels):
    print(f"  {lbl}  →  true Teff = {sampled.loc[i, 'y_true']:.0f} K")

## Predict with individual trees and collect GMM params

Since the model uses `target_transform: log`, individual tree predictions are in
log10-space. We inverse-transform them to Kelvin before plotting.

In [ ]:
X_sampled = sampled[features].values

# Per-tree predictions in log10 space, then inverse-transform to Kelvin
tree_preds_log = np.array([est.predict(X_sampled) for est in model.estimators_])
tree_preds_kelvin = 10 ** tree_preds_log  # (n_trees, n_objects)

# Extract stored GMM parameters
n_comp = 5
gmm_w = sampled[[f"gmm_weight_{k}" for k in range(n_comp)]].values
gmm_m = sampled[[f"gmm_mean_{k}" for k in range(n_comp)]].values
gmm_s = sampled[[f"gmm_sigma_{k}" for k in range(n_comp)]].values

print(f"Tree predictions shape: {tree_preds_kelvin.shape}  (n_trees × n_objects)")
print(f"GMM params shape: weights {gmm_w.shape}, means {gmm_m.shape}, sigmas {gmm_s.shape}")

## Plot: tree-prediction distributions per temperature bin

In [ ]:
target_info = {"name": "Temperature", "unit": "K", "short": "Teff"}

fig = plot_tree_distributions_grid(
    tree_preds_all=[tree_preds_kelvin[:, i] for i in range(len(sampled))],
    y_true_all=sampled["y_true"].tolist(),
    gmm_weights_all=gmm_w,
    gmm_means_all=gmm_m,
    gmm_sigmas_all=gmm_s,
    bin_labels=bin_labels,
    n_bins=50,
    target_info=target_info,
    suptitle=f"Per-tree prediction distributions — {MODEL_ID}",
    ncols=3,
)

subdir = "gaia_teff_corrected_log_ebv_optuna_validation"
save_figure(fig, f"{MODEL_ID}_tree_distributions.png", subdir)
plt.show()